In [108]:
import numpy as np
from numpy import random
import copy
from statistics import mean

LANDSCAPE_LENGTH = 100
NUM_AGENTS = 500
MAX_STEP = 20
STRATEGY_LENGTH = 3
TEAM_SIZE = 4
VERBOSE = False 
N_RERUNS = 100

# options for strategies: 
# Note:  "duplicates" here is not about agents but about elements within the heuristic
        # "NoRepeatsSorted": list of STRATEGY_LENGTH elements where each element is random choice between 1 and max_step
        # "AllowRepeatsUnsorted": list of STRATEGY_LENGTH elements where each element is random choice between 1 and max_step, repeats allowed, not sorted
        # "NoRepeatsUnsorted": list of STRATEGY_LENGTH elements where each element is random choice between 1 and max_step, repeats not allowed, not sorted

STRATEGY_TYPE = "NoRepeatsUnsorted"


In [117]:
random.seed(44)
LANDSCAPE = [
    {"location": i, 
     "value": random.randint(100)}
    for i in range(LANDSCAPE_LENGTH)
]

AGENTS = [
    {"strategy": [], "winnings": []}
    for _ in range(NUM_AGENTS)
]
for agent in AGENTS:

    if STRATEGY_TYPE == "AllowRepeatsUnsorted": 
        strategy = random.choice(range(1, MAX_STEP+1), STRATEGY_LENGTH)
        agent["strategy"] = strategy.tolist()

    elif STRATEGY_TYPE == "NoRepeatsUnsorted":
            strategy = random.choice(range(1, MAX_STEP+1), STRATEGY_LENGTH, replace=False)
            agent["strategy"] = strategy.tolist()

    elif STRATEGY_TYPE == "NoRepeatsSorted":
        strategy = random.choice(range(1, MAX_STEP+1), STRATEGY_LENGTH, replace=False)
        agent["strategy"] = sorted(strategy.tolist())

#starting_pos = random.randint(LANDSCAPE_LENGTH)



In [110]:
def climb(strategy, pos, verbose=VERBOSE):
    """Move until a full pass finds nothing better. Returns (final_pos, value of position).
    Note: agent checks heuristic from where they left off, not fresh."""
    n = len(strategy)
    i = 0         # which strategy element to try next
    misses = 0    # consecutive checks with no improvement

    while misses < n:
        candidate = (pos + strategy[i]) % LANDSCAPE_LENGTH

        if LANDSCAPE[candidate]["value"] > LANDSCAPE[pos]["value"]:
            if verbose:
                print("Moving from", pos, "to", candidate,
                      "with value", LANDSCAPE[candidate]["value"])
            pos = candidate
            misses = 0
        else:
            misses += 1

        i = (i + 1) % n

    return pos, LANDSCAPE[pos]["value"]

In [111]:
def run_solo(agents):
    agents = copy.deepcopy(agents)
    
    for start in range(len(LANDSCAPE)):
        for agent in agents:
            _, won = climb(agent["strategy"], start)
            agent["winnings"].append(won)
                
    return agents

In [115]:
agents = run_solo(AGENTS)

for agent in agents: 
    agent["avg_winnings"] = mean(agent["winnings"])

agents_sorted = sorted(agents,key=lambda a: mean(a["winnings"]), reverse=True)
for rank, agent in enumerate(agents_sorted[:10], start=1):
    print("Agent", agent["strategy"], "had average winnings", agent["avg_winnings"]) 
print(agents_sorted)

Agent [13, 20, 19] had average winnings 85.92
Agent [17, 10, 4] had average winnings 85.75
Agent [15, 17, 5] had average winnings 85.69
Agent [5, 17, 20] had average winnings 85.69
Agent [20, 5, 19] had average winnings 85.56
Agent [9, 10, 2] had average winnings 85.32
Agent [9, 2, 5] had average winnings 85
Agent [1, 19, 13] had average winnings 84.99
Agent [9, 6, 19] had average winnings 84.89
Agent [5, 2, 20] had average winnings 84.88
[{'strategy': [13, 20, 19], 'winnings': [99, 99, 68, 75, 89, 95, 88, 99, 99, 99, 74, 95, 87, 88, 99, 68, 75, 89, 85, 99, 99, 99, 99, 85, 95, 76, 88, 99, 85, 74, 60, 85, 85, 87, 87, 68, 85, 95, 85, 99, 99, 85, 85, 83, 95, 85, 99, 85, 85, 46, 53, 85, 70, 87, 78, 85, 99, 95, 85, 99, 85, 85, 46, 83, 85, 89, 83, 85, 99, 99, 49, 70, 87, 87, 85, 99, 83, 85, 89, 83, 85, 99, 99, 49, 70, 95, 85, 78, 99, 99, 74, 89, 87, 88, 99, 99, 99, 74, 95, 87], 'avg_winnings': 85.92}, {'strategy': [17, 10, 4], 'winnings': [89, 99, 99, 75, 73, 99, 88, 85, 76, 91, 75, 99, 99, 

In [104]:
import ollama

response = ollama.generate(
    model='llama3.1:8b', 
    prompt='You are an agent trying to find the maximum value in a list of 100 elements. The value of each element in the list is random. You can move only forward and the list loops around when you get to the end. I want you to develop a strategy called a heuristic. Your heuristic consists of the steps forward you want to take. It can only be 3 elements long, cannot include duplicates, and can have your elements in any order. The maximum steps forward you can take (the maximum element you can have in your heuristic) is 20. You will begin by checking the value of the location (your starting location plus the first element of your heuristic). If that is better than your current location, you will move there and check (that location plus your second heuristic). If not, check your starting location plus second heuristic. Repeat until you have exhausted your list of heuristics and cannot find a better location to move to. What is your heuristic?'
    ) 

print(response['response'])


KeyboardInterrupt: 

In [118]:
LLMagents = [
    {"strategy": [5, 3, 10], "winnings": []}    
]

LLMagents = run_solo(LLMagents)

for LLMagent in LLMagents: 
    LLMagent["avg_winnings"] = mean(LLMagent["winnings"])

LLMagents_sorted = sorted(LLMagents, key=lambda a: a["avg_winnings"], reverse=True)
for rank, agent in enumerate(LLMagents_sorted[:10], start=1):
    print("Agent", agent["strategy"], "had average winnings", agent["avg_winnings"]) 
print(LLMagents_sorted)

Agent [5, 3, 10] had average winnings 84.21
[{'strategy': [5, 3, 10], 'winnings': [99, 89, 99, 89, 83, 99, 89, 99, 96, 93, 99, 89, 99, 96, 93, 99, 85, 83, 96, 93, 96, 85, 83, 96, 93, 86, 72, 74, 86, 93, 75, 72, 74, 86, 92, 75, 92, 75, 73, 92, 75, 56, 66, 73, 56, 75, 56, 66, 73, 56, 96, 74, 96, 99, 62, 96, 99, 96, 81, 62, 96, 99, 96, 81, 74, 96, 99, 96, 99, 74, 96, 99, 62, 98, 50, 78, 98, 78, 98, 50, 78, 98, 50, 98, 50, 98, 98, 50, 98, 61, 99, 98, 99, 66, 61, 99, 89, 99, 66, 99], 'avg_winnings': 84.21}]
